# Contextual Compression and Knowledge Refinement for Advanced Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) systems are powerful, but their performance is often bottlenecked by the quality of the retrieved context. When a retriever pulls back large chunks of text that contain relevant information mixed with noise or tangential details, the Large Language Model (LLM) can become confused, leading to poor answers, hallucination, or an inability to focus on the core answer. This problem necessitates advanced techniques beyond simple vector similarity search. Contextual compression is a sophisticated method designed to solve this by not just retrieving *documents*, but rather extracting and refining the most salient, actionable passages from those documents before they reach the final prompt.

For developers building state-of-the-art AI agents using frameworks like LangGraph, mastering contextual compression is paramount. It elevates RAG from a simple lookup mechanism to a highly focused reasoning engine. By implementing compression, you ensure that the LLM receives a clean, distilled "summary of evidence" rather than an overwhelming wall of text. This dramatically improves prompt efficiency, reduces token usage costs, and most importantly, significantly boosts the factual accuracy and coherence of the generated output, making your agents more reliable for mission-critical applications.

In this notebook, we will explore the principles behind preparing diverse, dense knowledge bases (like those covering AI ethics, climate science, or genomics) and how to structure a pipeline that intelligently processes these sources. You will learn how to move beyond basic chunking strategies and implement methods that distill complex information into highly relevant context snippets, forming a critical building block for advanced multi-step reasoning graphs in LangGraph.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Understand the Limitations of Basic RAG:** Identify scenarios where simple vector retrieval fails due to noisy or overly verbose context chunks.
*   **Grasp Contextual Compression Principles:** Explain how and why distilling retrieved information into salient passages improves LLM performance.
*   **Handle Diverse Knowledge Domains:** Structure a data pipeline capable of ingesting, chunking, and embedding complex, multi-topic documents (e.g., scientific papers, economic reports).
*   **Prepare for Advanced Graph Architectures:** Understand how refined context snippets are essential inputs for sophisticated reasoning steps within LangGraph workflows.


### Setup and Imports

This cell imports necessary libraries and core components from LangChain and related packages. It brings in tools for handling environment variables (`dotenv`), defining data structures (`Document`), implementing retrieval logic (`BaseRetriever`, `InMemoryVectorStore`), constructing prompts (`ChatPromptTemplate`), parsing outputs (`StrOutputParser`), and initializing key models like OpenAI embeddings and chat models.


In [1]:
from dotenv import load_dotenv
from typing import Any
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings, ChatOpenAI


In [2]:
load_dotenv()

True

### Model Initialization

This cell initializes the core components for our RAG pipeline: an embedding model and a Large Language Model (LLM). `OpenAIEmbeddings` converts text into numerical vectors, while `ChatOpenAI` provides the generative AI capabilities needed for reasoning and response generation.


In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)


### Data Preparation: Loading Source Documents

This cell initializes a list of `Document` objects, simulating the loading of diverse source documents from various topics (AI, climate change, space, etc.). Each document contains rich text content and associated metadata, which is crucial for advanced retrieval-augmented generation (RAG) techniques like contextual compression.


In [5]:
docs = [
    Document(
        page_content=(
            "Artificial intelligence has made remarkable strides in natural language processing, "
            "with large language models now capable of generating human-quality text and code. "
            "Computer vision systems can identify objects in images with superhuman accuracy, "
            "powering applications from autonomous vehicles to medical imaging diagnostics. "
            "However, the rapid advancement of AI has raised significant ethical concerns about "
            "job displacement, algorithmic bias, and the concentration of power among a few tech companies."
        ),
        metadata={"topic": "artificial_intelligence"},
    ),
    Document(
        page_content=(
            "Global temperatures have risen by approximately 1.1 degrees Celsius since pre-industrial "
            "times, driven primarily by the burning of fossil fuels. The melting of polar ice caps has "
            "accelerated, contributing to rising sea levels that threaten coastal communities worldwide. "
            "Renewable energy adoption is growing rapidly, with solar and wind power becoming cheaper "
            "than coal in many regions. Governments are implementing carbon pricing mechanisms and "
            "investing in green infrastructure to meet Paris Agreement targets."
        ),
        metadata={"topic": "climate_change"},
    ),
    Document(
        page_content=(
            "NASA's Artemis program aims to return humans to the Moon by the mid-2020s, establishing "
            "a sustainable presence as a stepping stone to Mars. Private companies like SpaceX are "
            "developing reusable rocket technology that has dramatically reduced launch costs. "
            "The James Webb Space Telescope has captured unprecedented images of distant galaxies, "
            "revealing new insights about the early universe. Asteroid mining is being explored as a "
            "potential source of rare minerals needed for electronics manufacturing."
        ),
        metadata={"topic": "space_exploration"},
    ),
    Document(
        page_content=(
            "CRISPR gene editing technology has revolutionized medical genomics, enabling precise "
            "modifications to DNA sequences that were previously impossible. Researchers are using "
            "genomic data to develop personalized medicine approaches, tailoring treatments based on "
            "an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by "
            "COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare "
            "genetic disorders. Hospital information systems are increasingly integrating genomic data "
            "to support clinical decision-making at the point of care."
        ),
        metadata={"topic": "medicine"},
    ),
    Document(
        page_content=(
            "The global economy is navigating a period of high inflation driven by supply chain "
            "disruptions, energy price volatility, and post-pandemic demand surges. Central banks "
            "worldwide have raised interest rates aggressively to combat inflation, impacting housing "
            "markets and consumer spending. Cryptocurrency regulation is becoming a priority for "
            "financial authorities, with the EU's MiCA framework setting a global precedent. "
            "Trade tensions between major economies continue to reshape global supply chains, "
            "pushing companies toward nearshoring and diversification strategies."
        ),
        metadata={"topic": "economics"},
    ),
    Document(
        page_content=(
            "Quantum computing has reached a critical milestone with several companies demonstrating "
            "quantum advantage on specific computational tasks. Error correction remains the biggest "
            "challenge, as current quantum processors are highly susceptible to noise and decoherence. "
            "Quantum simulation of molecular structures could transform drug discovery by accurately "
            "modeling protein folding and chemical interactions. Major tech companies and governments "
            "are investing billions in quantum research, viewing it as essential for national security "
            "and economic competitiveness."
        ),
        metadata={"topic": "quantum_computing"},
    ),
]

print(f"Created {len(docs)} documents")



Created 6 documents


### Vectorstore Initialization and Retriever Setup

This cell initializes an in-memory vector store from the loaded documents (`docs`) using specified embeddings. It then converts this vector store into a retriever object, which is the primary mechanism used to fetch relevant document chunks based on similarity search (k=3).


In [6]:
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings) # 1. Create an in-memory vector store from the list of documents 'docs', using the provided embeddings model.
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # 2. Convert the vector store into a retriever object, configuring it to retrieve the top 3 (k=3) most relevant chunks.


### Custom Contextual Compression Extractor

This class, `CustomLLMExtractorChain`, implements a specialized mechanism for 'contextual compression.' It uses an LLM chain to read through provided documents and extract only the parts that are strictly relevant to a given query, ensuring the original text is preserved. This step significantly reduces the context window size while maintaining high informational fidelity.


In [7]:
# Custom LLM extractor that mirrors LLMChainExtractor using a Runnable chain internally
class CustomLLMExtractorChain:
    """Extracts relevant portions from documents using an LLM chain."""

    def __init__(self, llm_chain):
        # Initialize the class with the pre-built runnable LLM chain
        self.llm_chain = llm_chain

    @classmethod
    def from_llm(cls, llm):
        # This class method is used to instantiate CustomLLMExtractorChain
        # by setting up the necessary prompt and runnable chain.
        # Same prompt pattern used by LangChain's LLMChainExtractor
        prompt = ChatPromptTemplate.from_template(
            "Given the following question and context, extract any part of the context "
            "*AS IS* that is relevant to answer the question. If none of the context is "
            "relevant return NO_OUTPUT.\n\n"
            "Remember, *DO NOT* edit the extracted parts of the context.\n\n"
            "> Question: {question}\n"
            "> Context:\n>>>\n{context}\n>>>\n"
            "Extracted relevant parts:"
        )
        # create the chain: Prompt -> LLM call -> String Output Parser
        llm_chain = prompt | llm | StrOutputParser()
        return cls(llm_chain=llm_chain)

    def compress_documents(self, documents: list[Document], query: str) -> list[Document]:
        # This method iterates through the input documents and compresses them.
        compressed = []
        for doc in documents:
            # Invoke the LLM chain with the current document's content and the query
            result = self.llm_chain.invoke({"question": query, "context": doc.page_content})
            # Clean up whitespace from the result
            result = result.strip()
            # Check if the result is non-empty and not the designated 'NO_OUTPUT' signal
            if result and result != "NO_OUTPUT":
                # If relevant content is found, create a new Document object with only the extracted text
                compressed.append(Document(page_content=result, metadata=doc.metadata))
        return compressed


This custom retriever, `CustomContextualCompressionRetriever`, enhances standard retrieval by adding a crucial compression step. It first uses an existing base retriever to fetch documents and then passes these results through a specialized LLM extractor (`self.base_compressor`) to distill the most relevant context before returning them.


In [8]:
# Custom retriever that composes base retrieval with LLM compression
class CustomContextualCompressionRetriever(BaseRetriever):
    """Retriever that compresses documents using a custom LLM extractor chain."""

    base_retriever: BaseRetriever
    base_compressor: Any  # CustomLLMExtractorChain instance

    def _get_relevant_documents(self, query: str) -> list[Document]:
        # Step 1: retrieve documents from the base retriever
        docs = self.base_retriever.invoke(query)
        # Step 2: compress using the LLM extractor
        compressed_docs = self.base_compressor.compress_documents(docs, query)
        return compressed_docs


### Custom Contextual Compression Retrieval

This cell initializes and executes a specialized retriever, `CustomContextualCompressionRetriever`. This component enhances standard retrieval by first fetching relevant documents (`base_retriever`) and then using an LLM-powered compressor (`compressor`) to distill the retrieved content down to the most contextually relevant passages for the given query. This significantly improves RAG performance by reducing noise and focusing the final answer generation.


In [9]:
# Wire up the custom classes and run the same query as notebook 1
# Initialize the compressor using the LLM, which extracts key information.
compressor = CustomLLMExtractorChain.from_llm(llm)

# Create the specialized retriever that combines base retrieval with contextual compression.
custom_retriever = CustomContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor,
)

# Define the query we want to use for retrieval.
query = "How is CRISPR acting as a big enabler in creating personalized medicine?"

# Invoke the custom retriever with the query. This executes the full RAG pipeline:
# 1. Retrieve documents (base_retriever).
# 2. Compress/filter the retrieved content (custom_retriever logic).
results = custom_retriever.invoke(query)

# Iterate through and print the results, showing the topic and compressed content for each document.
for i, doc in enumerate(results):
    print(f"--- Result {i+1} [{doc.metadata.get('topic')}] ---")
    print(doc.page_content)
    print()


--- Result 1 [medicine] ---
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile.

